In [1]:
// load the formatting setup and the style sheet
#load "../setup.fsx"
HTML(System.IO.File.ReadAllText "../style.css")

The below script needs to be able to find the current output cell; this is an easy method to get it.

# The Twin-Primes Problem

The notebook answers the following question: Could we decide the question whether there are infinitely many twin primes if the *halting problem* was solvable?

The function `divisors` takes one argument:
- `k` is a positive natural number.

The function returns the set of all natural numbers `t` from the range $\{1, \cdots, k\}$ that divide `k` evenly.

In [2]:
let divisors (k: int) : Set<int> =
    set [ for t in 1 .. k do if k % t = 0 then t ]

The function `isPrime` takes one argument:
- `k` is a positive natural number.

The function returns `true` if `k` is a prime number and `false` otherwise.  A number $k$ is prime if $k \not= 1$ and the set of its divisors is $\{1, k\}$.

In [3]:
let isPrime (k: int) : bool =
    k <> 1 && divisors k = set [1; k]

The function `existsPrimePair` takes one argument:
- `n` is a natural number.

The function checks whether there is a pair $\langle p, p+2 \rangle$ of primes such that $p \geq n$.  In case a pair of primes is found, the first such pair is returned.
If there is no such pair, the function call `existsPrimePair n` does not terminate.

**Implementation:** The search is done by the local recursive function `search`, which tests the numbers $p = n, n+1, n+2, \cdots$ one after the other.  Since the recursive call of `search` is a *tail call*, the recursion is translated into a loop by the compiler.

In [4]:
let existsPrimePair (n: int) : int * int =
    let rec search (p: int) : int * int =
        if isPrime p && isPrime (p + 2) then
            (p, p + 2)
        else
            search (p + 1)
    search n

The magic command `#!time` at the beginning of a cell measures the time needed to execute the cell.

In [5]:
#!time
existsPrimePair 10_000_000

(10000139, 10000141)

Wall time: 3121.4174ms

To continue, we *assume* that there is a function `stops` that solves the halting problem.  This function takes two arguments:
- `f` is a function,
- `a` is an argument for the function `f`.

The function call `stops f a` returns
- `1` if the evaluation of `f a` terminates, and
- `0` if the evaluation of `f a` does not terminate.

Of course, such a function does not exist.  However, in F# every function has to be defined before it can be used.  Therefore, we define `stops` as a function that raises an exception.  The definition only serves to tell the compiler the type of `stops`: the type variables `'a` and `'b` specify that `stops` can be applied to any function `f` of type `'a -> 'b` and to any argument `a` of type `'a`.

In [6]:
let stops (f: 'a -> 'b) (a: 'a) : int =
    failwith "The halting problem is not solvable!"

The function `findCounterExample` takes one argument:
- `n` is a natural number.

The function tries to find a natural number $b \geq n$ such that there are no twin primes bigger than $b$.  If such a number $b$ is found, it is returned.
In order to do so, it uses the function `stops` that solves the *halting problem*: if `stops existsPrimePair b` returns `0`, then there is no pair of twin primes beyond `b`.

If there is no such number $b$, the function call `findCounterExample n` does not
terminate.

In [7]:
let findCounterExample (n: int) : int =
    let rec search (b: int) : int =
        if stops existsPrimePair b = 0 then  // no prime pair beyond b
            b
        else
            search (b + 1)
    search n

The function `isTwinPrimeConjectureTrue` takes no arguments.
The function call `isTwinPrimeConjectureTrue ()` returns `true`
if and only if the set of twin primes is infinite.  This is the case if and only if the function call
`findCounterExample 1` does not terminate.

In [8]:
let isTwinPrimeConjectureTrue () : bool =
    stops findCounterExample 1 = 0

Hence, if the halting problem were solvable, we could decide whether there are infinitely many twin primes.